# RedZone world model: scaling ladder on a Colab A100, recipe E (EXP-107)

Personal use only: plan B data, so grades.json carries "personal-use, not for commercial release".
Recipe E, the EXP-109 winner (2.07x faster to EXP-103's loss, best in every league): Muon + warmup-stable-decay +
QK-norm, ReLU², untied head, logit soft-cap 15, value embeddings + torch.compile, lr 6e-4. Same portable trainer as the
Kaggle TPU notebook (scripts/nfm_train.py: CUDA, bf16, packing). Every point trains on the same budget, min(1.2B tokens, 4 epochs of the unique train tokens), with dropout 0.1 and
micro-batches of 4-16 (grad accumulation to 32)
(vocab v2: nflverse NFL, NBA.com cdn NBA, Retrosheet MLB). The table reports NBA first, then NFL.

Default LADDER = 18m, 60m, 150m (150m is the Saturday-size point). Runtime > Change runtime type > **A100 GPU**
(High-RAM), then **Run all**. A smoke pass (50 steps, a checkpoint and a tiny grade per point, ~5 min after the data build) runs first and
stops the notebook if anything would crash. About 4.5 h (~60 CU): data ~40 min, 18m ~0.5 h, 60m ~1 h, 150m ~2.3 h. If the runtime
restarts but keeps its disk, Run all again: finished points are skipped and the running one resumes from its 30-min
checkpoint. The 300m point (~4 h, ~55 CU) is a separate run once these three are graded.


In [ ]:
PIN = "3134544496ca655b9b32b1a185b93b6c698f3f96"   # kaggle-inputs commit: factory code pack (nfm_train.py + noggin_tokens_v2.py)
REPO = "BouncySlime1215/redzone-everything"
LEAGUES = "nfl,nba,mlb"
LADDER = ["18m", "60m", "150m"]   # then ["300m"]; ["moe-60m"] is EXP-108
EXPS = {"moe-60m": "EXP-108"}
MINUTES = {"18m": 60, "60m": 120, "150m": 240, "300m": 420, "moe-60m": 150}   # caps above the budget: WSD must reach its decay
CTX, BATCH, TRAIN_TOKENS = 2048, 32, 1.2e9
RECIPE_E = ["--muon", "--sched", "wsd", "--qk-norm", "--relu2", "--untie", "--softcap", "15", "--value-emb",
            "--compile", "--lr", "6e-4", "--warmup", "200",
            "--max-epochs", "4", "--dropout", "0.1"]         # run 1 memorized at 5.5 epochs (train 0.28 vs valid 0.50)
MICRO = {"18m": 16, "60m": 8, "150m": 4, "300m": 4, "moe-60m": 8}   # grad accumulation to BATCH (run 1 OOM on 40 GB)
ENV = {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
RUN, CODE = "/content/rz", "/content/rz/code"
CU_PER_H = {"A100": 13.0, "L4": 5.0, "T4": 2.0}


In [ ]:
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime > Change runtime type > A100 GPU, then Run all again")
GPU = torch.cuda.get_device_name(0)
RATE = next((v for k, v in CU_PER_H.items() if k in GPU), None)
print("GPU:", GPU, "| CU/h:", RATE)


In [ ]:
import json, os, subprocess, sys, tarfile, time, urllib.request
T_START = time.time()
os.makedirs(CODE, exist_ok=True)
if not os.path.exists(f"{CODE}/rz/scripts/nfm_train.py"):
    for i, ref in enumerate((PIN, PIN[:12], PIN, PIN[:12])):     # the raw CDN can lag a fresh push for a minute
        url = f"https://raw.githubusercontent.com/{REPO}/{ref}/factory/rz_factory_code.tar.gz"
        try:
            print("downloading", url)
            urllib.request.urlretrieve(url, f"{RUN}/code.tar.gz")
            break
        except Exception as e:
            print(" ", type(e).__name__, e)
            time.sleep(30 * i)
    else:
        raise SystemExit("could not download the code pack; Run all again in a minute")
    with tarfile.open(f"{RUN}/code.tar.gz") as t:
        t.extractall(CODE, filter="data")
print("code:", sorted(os.listdir(f"{CODE}/rz")))


In [ ]:
def run(preset, extra=(), smoke=False):
    out = f"{RUN}/{'smoke' if smoke else 'nfm'}/{preset}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/grades.json"):
        print(preset, "done already"); return
    cmd = [sys.executable, "-u", f"{CODE}/rz/scripts/nfm_train.py", "--out", out, "--ckpt-dir", out,
           "--data-dir", f"{RUN}/data", "--build-vocab", "2", "--leagues", LEAGUES, "--preset", preset,
           "--ctx", str(CTX), "--batch", str(BATCH), "--train-tokens", str(TRAIN_TOKENS),
           "--minutes", str(MINUTES[preset]), "--exp", EXPS.get(preset, "EXP-107"), *RECIPE_E, "--micro-batch", str(MICRO[preset]), *extra]
    if smoke:
        cmd += ["--max-steps", "50", "--ckpt-min", "0.2", "--eval-every", "25", "--grade-games", "5"]
    with open(f"{out}/train.log", "a") as log:
        p = subprocess.Popen(cmd, cwd=RUN, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                             env={**os.environ, "PYTHONUNBUFFERED": "1", **ENV})
        for line in p.stdout:
            print(line, end=""); log.write(line)
        rc = p.wait()
        print(preset, "smoke" if smoke else "", "exit code:", rc)
    if rc != 0 or not os.path.exists(f"{out}/grades.json"):
        print("".join(open(f"{out}/train.log").readlines()[-40:]))
        if smoke:
            raise SystemExit(f"SMOKE FAILED for {preset}: fix before the long run (log tail above)")

# SMOKE (~5 min after the data build): every point runs 50 steps, an in-loop checkpoint save and a 5-game grade first,
# so an OOM or a crash shows up now, not hours in. A failed smoke stops the notebook before the long runs.
for preset in LADDER:
    if not os.path.exists(f"{RUN}/nfm/{preset}/grades.json"):
        run(preset, smoke=True)
print(f"smoke passed for {LADDER} at {(time.time() - T_START) / 60:.0f} min; long runs start now")
for preset in LADDER:
    run(preset)


In [ ]:
rows = []
for preset in LADDER:
    gp = f"{RUN}/nfm/{preset}/grades.json"
    if os.path.exists(gp):
        g = json.load(open(gp))
        v = g.get("valid_loss_by_league", {})
        t = {lg: (g.get("test_by_league", {}).get(lg, {}).get("next_kind_acc") or {}).get("value") for lg in ("nba", "nfl")}
        rows.append((preset, g["params"]["non_embedding_active"], g["best_valid_loss"], v.get("nba"), v.get("nfl"),
                     v.get("mlb"), t["nba"], t["nfl"], g["throughput"]["tokens_per_s"], g["steps"], g["planned_steps"],
                     g["train_minutes"], g["license"]))
print("preset | non-emb params | best valid | valid NBA | valid NFL | valid MLB | test next-kind NBA | NFL | tok/s | steps | planned | train min | license")
for r in rows:
    print(" | ".join(str(x) for x in r))
print(f"session wall clock: {(time.time() - T_START) / 3600:.2f} h")
json.dump(rows, open(f"{RUN}/ladder_summary.json", "w"))
print("CU spent: ~", round((time.time() - T_START) / 3600 * (RATE or 0), 1))
